In [1]:
# =====================================================================
# APA-DDoS — REVISION EXPERIMENTS (answers Reviewer comments 1, 2, 3)
#
#   E0  ALL_FEATURES      : same inputs as the submitted paper (reproduces Fig. 22)
#   E1  NO_IDENTIFIERS    : IPs, ports, frame.time, tcp.seq, tcp.ack removed
#   E2  NO_ID_NO_PUSH     : E1 + tcp.flags.push removed (label-proxy ablation)
#
#   Each experiment is run with
#     - "constrained"   models (hyper-parameters exactly as in Section 3.2)
#     - "unconstrained" DT / RF / XGBoost (Reviewer comment 3)
#
#   Diagnostics (Reviewer comment 2):
#     - list of every input feature actually used (-> new table in paper)
#     - % of test rows whose categorical value was UNSEEN in training
#     - depth-1 decision stump per feature on DDoS-ACK vs DDoS-PSH-ACK
#     - train vs test accuracy + top feature importances for tree models
#
#   Outputs are written to ./revision_outputs/
# =====================================================================

import os, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder, label_binarize
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, roc_auc_score)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
DATA_PATH = "/kaggle/input/datasets/yashwanthkumbam/apaddos-dataset/APA-DDoS-Dataset/APA-DDoS-Dataset.csv"
OUT = "revision_outputs"
SEED = 42
RUN_SVM = True            # set False for a quick run (SVM is the slowest model)
os.makedirs(OUT, exist_ok=True)

# Identifier / session fields: tied to HOW the testbed generated each attack,
# not to attack behaviour. Matched on lower-cased column names.
IDENTIFIER_COLS = ["ip.src", "ip.dst", "tcp.srcport", "tcp.dstport",
                   "frame.time", "tcp.seq", "tcp.ack"]
# Header flag that defines ACK vs PSH-ACK flooding by construction.
PROXY_COLS = ["tcp.flags.push"]

EXPERIMENTS = {
    "E0_ALL_FEATURES": [],
    "E1_NO_IDENTIFIERS": IDENTIFIER_COLS,
    "E2_NO_ID_NO_PUSH": IDENTIFIER_COLS + PROXY_COLS,
}


def constrained_models(n_classes):
    """Hyper-parameters exactly as reported in Section 3.2 of the manuscript."""
    m = {
        # lbfgs = true multinomial (softmax), matching Eq. (11).
        # NOTE: liblinear (as written in the paper) is one-vs-rest, NOT softmax.
        "Logistic Regression": LogisticRegression(solver="lbfgs", C=0.03, max_iter=1000,
                                                  random_state=SEED),
        "SVM": SVC(kernel="rbf", C=1.0, probability=True, random_state=SEED),
        "KNN": KNeighborsClassifier(n_neighbors=5, metric="manhattan", weights="distance",
                                    n_jobs=-1),
        "Decision Tree": DecisionTreeClassifier(criterion="gini", max_depth=8,
                                                max_leaf_nodes=11, random_state=SEED),
        "Random Forest": RandomForestClassifier(n_estimators=20, criterion="entropy",
                                                max_depth=10, min_samples_split=100,
                                                min_samples_leaf=20, max_features="sqrt",
                                                random_state=SEED, n_jobs=-1),
        "XGBoost": XGBClassifier(eval_metric="mlogloss", random_state=SEED, n_jobs=-1),
    }
    if not RUN_SVM:
        m.pop("SVM")
    return m


def unconstrained_trees():
    """Reviewer comment 3: same models with capacity limits removed."""
    return {
        "Decision Tree (unconstrained)": DecisionTreeClassifier(random_state=SEED),
        "Random Forest (unconstrained)": RandomForestClassifier(n_estimators=200,
                                                                random_state=SEED, n_jobs=-1),
        "XGBoost (500 trees, depth 10)": XGBClassifier(n_estimators=500, max_depth=10,
                                                       eval_metric="mlogloss",
                                                       random_state=SEED, n_jobs=-1),
    }


# ---------------------------------------------------------------------
# DATA
# ---------------------------------------------------------------------
def load():
    df = pd.read_csv(DATA_PATH, low_memory=False)
    df.columns = df.columns.str.strip().str.lower()
    print("Columns in file:", list(df.columns))
    print("Shape:", df.shape)
    n_dup = df.duplicated().sum()
    print("Exact duplicate rows:", n_dup)
    return df


def split(df, drop_cols):
    y_raw = df["label"]
    X = df.drop(columns=["label"])
    missing = [c for c in drop_cols if c not in X.columns]
    if missing:
        print("  WARNING - columns to drop not found (check names):", missing)
    X = X.drop(columns=[c for c in drop_cols if c in X.columns])

    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.30,
                                              random_state=SEED, stratify=y)
    X_tr, X_te = X_tr.copy(), X_te.copy()

    # --- categorical encoding (fit on train only) + UNSEEN-value audit ---
    cat_cols = list(X_tr.select_dtypes(include=["object", "category"]).columns)
    unseen = {}
    if cat_cols:
        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        X_tr[cat_cols] = enc.fit_transform(X_tr[cat_cols].astype(str))
        X_te[cat_cols] = enc.transform(X_te[cat_cols].astype(str))
        for c in cat_cols:
            unseen[c] = float((X_te[c] == -1).mean() * 100)

    X_tr = X_tr.apply(pd.to_numeric, errors="coerce")
    X_te = X_te.apply(pd.to_numeric, errors="coerce")
    med = X_tr.median()
    X_tr, X_te = X_tr.fillna(med), X_te.fillna(med)

    const = [c for c in X_tr.columns if X_tr[c].nunique() <= 1]
    X_tr, X_te = X_tr.drop(columns=const), X_te.drop(columns=const)

    feats = list(X_tr.columns)
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_te_s = sc.transform(X_te)
    info = {"features": feats, "categorical": cat_cols, "constant_dropped": const,
            "unseen_pct": unseen}
    return X_tr_s, X_te_s, y_tr, y_te, le.classes_, info


# ---------------------------------------------------------------------
# METRICS
# ---------------------------------------------------------------------
def evaluate(name, model, X_tr, X_te, y_tr, y_te, classes):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    t_train = time.time() - t0
    y_pred = model.predict(X_te)
    row = {"Model": name,
           "Train Acc": accuracy_score(y_tr, model.predict(X_tr)) * 100,
           "Test Acc": accuracy_score(y_te, y_pred) * 100}
    for avg in ["macro", "micro", "weighted"]:
        p, r, f, _ = precision_recall_fscore_support(y_te, y_pred, average=avg,
                                                     zero_division=0)
        row[f"P {avg}"], row[f"R {avg}"], row[f"F1 {avg}"] = p * 100, r * 100, f * 100
    try:
        prob = model.predict_proba(X_te)
        row["ROC-AUC (OvR macro)"] = roc_auc_score(
            label_binarize(y_te, classes=np.arange(len(classes))), prob,
            multi_class="ovr", average="macro")
    except Exception as e:
        print("  AUC failed:", e)
        row["ROC-AUC (OvR macro)"] = np.nan
    row["Train Time (s)"] = t_train
    return row, confusion_matrix(y_te, y_pred)


def plot_cms(cms, classes, title, path):
    n = len(cms)
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.3 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (name, (cm, acc)) in zip(axes, cms.items()):
        ax.imshow(cm, cmap="Blues")
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black",
                        fontsize=10, fontweight="bold")
        ax.set_xticks(range(len(classes)))
        ax.set_xticklabels(classes, rotation=20, ha="right", fontsize=8)
        ax.set_yticks(range(len(classes)))
        ax.set_yticklabels(classes, fontsize=8)
        ax.set_title(f"{name} (Acc = {acc:.2f}%)", fontsize=10, fontweight="bold")
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)


# ---------------------------------------------------------------------
# DIAGNOSTICS (Reviewer comment 2)
# ---------------------------------------------------------------------
def stump_audit(X_tr, X_te, y_tr, y_te, classes, feats):
    """Depth-1 tree on ONE feature, ACK vs PSH-ACK only.
    ~100% for a single feature = that feature alone defines the label."""
    idx_ack = [i for i, c in enumerate(classes) if "psh" not in c.lower() and "ack" in c.lower()]
    idx_psh = [i for i, c in enumerate(classes) if "psh" in c.lower()]
    if not idx_ack or not idx_psh:
        print("  Could not identify ACK / PSH-ACK classes:", classes)
        return pd.DataFrame()
    keep = [idx_ack[0], idx_psh[0]]
    mtr, mte = np.isin(y_tr, keep), np.isin(y_te, keep)
    rows = []
    for j, f in enumerate(feats):
        s = DecisionTreeClassifier(max_depth=1, random_state=SEED)
        s.fit(X_tr[mtr][:, [j]], y_tr[mtr])
        rows.append({"Feature": f,
                     "Stump Train Acc (ACK vs PSH-ACK)": s.score(X_tr[mtr][:, [j]], y_tr[mtr]) * 100,
                     "Stump Test Acc (ACK vs PSH-ACK)": s.score(X_te[mte][:, [j]], y_te[mte]) * 100})
    return pd.DataFrame(rows).sort_values("Stump Test Acc (ACK vs PSH-ACK)", ascending=False)


def importances(model, feats, k=8):
    if hasattr(model, "feature_importances_"):
        imp = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False)
        return imp.head(k)
    return None


# ---------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------
if __name__ == "__main__":
    df = load()
    print("\nClass distribution:\n", df["label"].value_counts())

    all_rows, feature_tables = [], []

    for exp, drop_cols in EXPERIMENTS.items():
        print("\n" + "=" * 70 + f"\n{exp}   (dropped: {drop_cols or 'nothing'})\n" + "=" * 70)
        X_tr, X_te, y_tr, y_te, classes, info = split(df, drop_cols)

        print(f"  {len(info['features'])} input features used:", info["features"])
        print("  Constant columns dropped:", info["constant_dropped"])
        if info["unseen_pct"]:
            print("  % of TEST rows with a categorical value never seen in TRAIN:")
            for c, v in info["unseen_pct"].items():
                print(f"     {c:<20s} {v:6.2f}%")
        feature_tables.append(pd.DataFrame({"Experiment": exp,
                                            "Feature": info["features"]}))

        # ---- stump audit ----
        st = stump_audit(X_tr, X_te, y_tr, y_te, classes, info["features"])
        if not st.empty:
            st.to_csv(f"{OUT}/{exp}_single_feature_stumps.csv", index=False)
            print("\n  Top single-feature stumps (ACK vs PSH-ACK):")
            print(st.head(6).to_string(index=False))

        # ---- constrained + unconstrained models ----
        model_sets = {"constrained": constrained_models(len(classes)),
                      "unconstrained": unconstrained_trees()}
        for setting, models in model_sets.items():
            cms = {}
            for name, model in models.items():
                print(f"\n  -> [{setting}] {name}")
                row, cm = evaluate(name, model, X_tr, X_te, y_tr, y_te, classes)
                row.update({"Experiment": exp, "Setting": setting,
                            "N features": len(info["features"])})
                all_rows.append(row)
                cms[name] = (cm, row["Test Acc"])
                print(f"     train acc {row['Train Acc']:.2f}% | test acc {row['Test Acc']:.2f}% "
                      f"| macro F1 {row['F1 macro']:.2f}% | AUC {row['ROC-AUC (OvR macro)']:.4f}")
                imp = importances(model, info["features"])
                if imp is not None:
                    print("     top importances:", ", ".join(f"{k}={v:.3f}" for k, v in imp.items()))
            plot_cms(cms, list(classes), f"APA-DDoS — {exp} — {setting}",
                     f"{OUT}/{exp}_{setting}_confusion.png")

    res = pd.DataFrame(all_rows)
    cols = ["Experiment", "Setting", "N features", "Model", "Train Acc", "Test Acc",
            "P macro", "R macro", "F1 macro", "P micro", "R micro", "F1 micro",
            "P weighted", "R weighted", "F1 weighted", "ROC-AUC (OvR macro)", "Train Time (s)"]
    res = res[cols].round(4)
    res.to_csv(f"{OUT}/all_results.csv", index=False)
    pd.concat(feature_tables).to_csv(f"{OUT}/features_used_per_experiment.csv", index=False)

    print("\n\nSUMMARY (test accuracy %)")
    print(res.pivot_table(index="Model", columns="Experiment", values="Test Acc").round(2))
    print(f"\nAll outputs saved in ./{OUT}/")

Columns in file: ['ip.src', 'ip.dst', 'tcp.srcport', 'tcp.dstport', 'ip.proto', 'frame.len', 'tcp.flags.syn', 'tcp.flags.reset', 'tcp.flags.push', 'tcp.flags.ack', 'ip.flags.mf', 'ip.flags.df', 'ip.flags.rb', 'tcp.seq', 'tcp.ack', 'frame.time', 'packets', 'bytes', 'tx packets', 'tx bytes', 'rx packets', 'rx bytes', 'label']
Shape: (151200, 23)
Exact duplicate rows: 0

Class distribution:
 label
Benign          75600
DDoS-PSH-ACK    37800
DDoS-ACK        37800
Name: count, dtype: int64

E0_ALL_FEATURES   (dropped: nothing)
  12 input features used: ['ip.src', 'tcp.srcport', 'frame.len', 'tcp.flags.push', 'ip.flags.df', 'frame.time', 'packets', 'bytes', 'tx packets', 'tx bytes', 'rx packets', 'rx bytes']
  Constant columns dropped: ['ip.dst', 'tcp.dstport', 'ip.proto', 'tcp.flags.syn', 'tcp.flags.reset', 'tcp.flags.ack', 'ip.flags.mf', 'ip.flags.rb', 'tcp.seq', 'tcp.ack']
  % of TEST rows with a categorical value never seen in TRAIN:
     ip.src                 0.00%
     ip.dst         

In [2]:
# =====================================================================
# APA-DDoS — LEAKAGE-FREE FEATURE PIPELINE  (train / validation / test)
#
#   Split: 60% train / 20% validation / 20% test (stratified, seed 42)
#     - train      : fit encoders, screens, models
#     - validation : feature screening + hyper-parameter selection
#     - test       : touched ONCE, for the final numbers in the paper
#
#   Feature audit (all decisions made on TRAIN / VALIDATION only):
#     1. Identifier removal   : IPs, ports, timestamp, seq/ack numbers
#                               + auto-detect: text columns with many unique values
#     2. Constant / near-constant removal (>= 99.9% one value)
#     3. Behavioural feature engineering (ratios / average sizes)
#     4. Redundancy removal   : |Pearson r| >= 0.98 with an earlier feature
#     5. Label-proxy screen   : a depth-1 tree on ONE feature, for every
#                               pair of classes. >= 99.5% on validation =
#                               the feature alone defines that class pair.
#     6. Mutual-information ranking of the surviving features
#
#   Two variants are trained and compared:
#     A_LEAKAGE_FREE   : steps 1-4 (identifiers removed; header flags kept)
#     B_BEHAVIOUR_ONLY : A + remove single-feature proxies of the two ATTACK
#                        classes (tcp.flags.push) -> traffic behaviour only
#
#   Models: LR, SVM, KNN, DT, RF, XGBoost, each tuned on the validation set.
#   Outputs -> ./leakfree_outputs/
# =====================================================================

import os, time, itertools, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             roc_auc_score, confusion_matrix)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
DATA_PATH = "/kaggle/input/datasets/yashwanthkumbam/apaddos-dataset/APA-DDoS-Dataset/APA-DDoS-Dataset.csv"
OUT = "leakfree_outputs"
SEED = 42
RUN_SVM = True
SVM_TUNE_SAMPLE = 20000          # SVM is tuned on a stratified subsample (speed)
IDENTIFIER_COLS = ["ip.src", "ip.dst", "tcp.srcport", "tcp.dstport",
                   "frame.time", "tcp.seq", "tcp.ack"]
AUTO_ID_UNIQUE_RATIO = 0.05      # text column with > 5% unique values -> identifier
NEAR_CONSTANT = 0.999
CORR_THRESHOLD = 0.98
PROXY_THRESHOLD = 99.5           # % validation accuracy of a single-feature stump
os.makedirs(OUT, exist_ok=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)


# ---------------------------------------------------------------------
# MODELS + SEARCH GRIDS (selected on validation macro-F1)
# ---------------------------------------------------------------------
def model_grid():
    g = {
        "Logistic Regression": (lambda **p: LogisticRegression(max_iter=2000, random_state=SEED, **p),
                                {"C": [0.01, 0.1, 1, 10]}),
        "SVM": (lambda **p: SVC(kernel="rbf", probability=True, random_state=SEED, **p),
                {"C": [0.1, 1, 10], "gamma": ["scale", 0.1]}),
        "KNN": (lambda **p: KNeighborsClassifier(n_jobs=-1, **p),
                {"n_neighbors": [3, 5, 11, 21], "weights": ["uniform", "distance"],
                 "metric": ["euclidean", "manhattan"]}),
        "Decision Tree": (lambda **p: DecisionTreeClassifier(random_state=SEED, **p),
                          {"max_depth": [4, 8, 16, None], "min_samples_leaf": [1, 10, 50]}),
        "Random Forest": (lambda **p: RandomForestClassifier(random_state=SEED, n_jobs=-1, **p),
                          {"n_estimators": [100, 300], "max_depth": [10, None],
                           "min_samples_leaf": [1, 5]}),
        "XGBoost": (lambda **p: XGBClassifier(eval_metric="mlogloss", random_state=SEED,
                                              n_jobs=-1, **p),
                    {"n_estimators": [200, 500], "max_depth": [4, 8],
                     "learning_rate": [0.05, 0.1]}),
    }
    if not RUN_SVM:
        g.pop("SVM")
    return g


# ---------------------------------------------------------------------
# DATA
# ---------------------------------------------------------------------
def load():
    df = pd.read_csv(DATA_PATH, low_memory=False)
    df.columns = df.columns.str.strip().str.lower()
    print("Shape:", df.shape)
    print("Class distribution:\n", df["label"].value_counts(), "\n")
    return df


def three_way_split(df):
    idx = np.arange(len(df))
    tr, tmp = train_test_split(idx, test_size=0.40, stratify=df["label"], random_state=SEED)
    va, te = train_test_split(tmp, test_size=0.50, stratify=df["label"].iloc[tmp],
                              random_state=SEED)
    return tr, va, te


def engineer(X):
    """Behavioural features derived from traffic volume (no identifiers)."""
    X = X.copy()
    def safe_div(a, b):
        return np.where(b > 0, a / np.maximum(b, 1e-9), 0.0)
    pairs = {
        "avg_pkt_size":      ("bytes", "packets"),
        "tx_avg_pkt_size":   ("tx bytes", "tx packets"),
        "rx_avg_pkt_size":   ("rx bytes", "rx packets"),
        "tx_rx_pkt_ratio":   ("tx packets", "rx packets"),
        "tx_rx_byte_ratio":  ("tx bytes", "rx bytes"),
        "frame_to_avg_size": ("frame.len", "avg_pkt_size"),
    }
    for new, (a, b) in pairs.items():
        if a in X.columns and b in X.columns:
            X[new] = safe_div(X[a].astype(float), X[b].astype(float))
    return X


# ---------------------------------------------------------------------
# FEATURE AUDIT
# ---------------------------------------------------------------------
def audit_features(df, tr, va, classes_enc, y):
    audit = []                                   # (feature, status, reason)
    X = df.drop(columns=["label"])

    # 1. identifiers
    ids = [c for c in IDENTIFIER_COLS if c in X.columns]
    for c in X.columns:
        if c not in ids and X[c].dtype == object:
            if X[c].iloc[tr].nunique() / len(tr) > AUTO_ID_UNIQUE_RATIO:
                ids.append(c)
    for c in ids:
        audit.append((c, "removed", "identifier (host/port/time/session field)"))
    X = X.drop(columns=ids)

    # remaining text columns -> numeric codes fitted on train
    for c in X.select_dtypes(include="object").columns:
        cats = {v: i for i, v in enumerate(pd.unique(X[c].iloc[tr]))}
        X[c] = X[c].map(cats).fillna(-1)
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.iloc[tr].median())

    # 2. constant / near-constant (on train)
    for c in list(X.columns):
        top = X[c].iloc[tr].value_counts(normalize=True).iloc[0]
        if top >= NEAR_CONSTANT:
            audit.append((c, "removed", f"constant/near-constant on train ({top*100:.2f}% one value)"))
            X = X.drop(columns=c)

    # 3. engineered behavioural features
    before = set(X.columns)
    X = engineer(X)
    for c in X.columns:
        if c not in before:
            audit.append((c, "added", "engineered behavioural feature"))

    # 4. redundancy (on train)
    corr = X.iloc[tr].corr().abs()
    cols = list(X.columns); drop = set()
    for i, a in enumerate(cols):
        if a in drop:
            continue
        for b in cols[i + 1:]:
            if b not in drop and corr.loc[a, b] >= CORR_THRESHOLD:
                drop.add(b)
                audit.append((b, "removed", f"redundant with {a} (|r| = {corr.loc[a, b]:.3f})"))
    X = X.drop(columns=list(drop))

    # 5. label-proxy screen: 1 feature, every class pair (fit train, score validation)
    proxy_rows = []
    for (i, j) in itertools.combinations(range(len(classes_enc)), 2):
        mtr = np.isin(y[tr], [i, j]); mva = np.isin(y[va], [i, j])
        for c in X.columns:
            s = DecisionTreeClassifier(max_depth=1, random_state=SEED)
            s.fit(X[[c]].iloc[tr][mtr], y[tr][mtr])
            acc = s.score(X[[c]].iloc[va][mva], y[va][mva]) * 100
            proxy_rows.append({"Feature": c, "Class pair": f"{classes_enc[i]} vs {classes_enc[j]}",
                               "Stump val acc (%)": acc})
    proxy = pd.DataFrame(proxy_rows)
    proxy["Single-feature proxy"] = proxy["Stump val acc (%)"] >= PROXY_THRESHOLD

    # 6. mutual information (train)
    rng = np.random.RandomState(SEED)
    sub = rng.choice(tr, size=min(30000, len(tr)), replace=False)
    mi = mutual_info_classif(X.iloc[sub], y[sub], random_state=SEED)
    mi = pd.DataFrame({"Feature": X.columns, "MI (train)": mi}).sort_values("MI (train)", ascending=False)

    for c in X.columns:
        if not any(a[0] == c and a[1] == "added" for a in audit):
            audit.append((c, "kept", "passed screens"))
    return X, pd.DataFrame(audit, columns=["Feature", "Status", "Reason"]), proxy, mi


def attack_pair_proxies(proxy, classes):
    """Features that alone separate the two ATTACK classes (not benign)."""
    benign = [c for c in classes if "benign" in c.lower()]
    mask = proxy["Single-feature proxy"]
    if benign:
        mask &= ~proxy["Class pair"].str.contains(benign[0])
    return sorted(proxy.loc[mask, "Feature"].unique())


# ---------------------------------------------------------------------
# TRAIN / TUNE / EVALUATE
# ---------------------------------------------------------------------
def evaluate(model, X, y, n_cls):
    pred = model.predict(X)
    prob = model.predict_proba(X)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    try:
        auc = roc_auc_score(label_binarize(y, classes=np.arange(n_cls)), prob,
                            multi_class="ovr", average="macro")
    except Exception:
        auc = np.nan
    return {"acc": accuracy_score(y, pred) * 100, "P": p * 100, "R": r * 100,
            "F1": f * 100, "F1w": f1_score(y, pred, average="weighted") * 100,
            "AUC": auc}, pred


def plot_cms(cms, classes, title, path):
    n = len(cms); cols = 3; rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.6 * cols, 4.3 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (name, cm) in zip(axes, cms.items()):
        acc = np.trace(cm) / cm.sum() * 100
        ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max())
        for a in range(cm.shape[0]):
            for b in range(cm.shape[1]):
                ax.text(b, a, f"{cm[a, b]:,}", ha="center", va="center", fontsize=10,
                        fontweight="bold", color="white" if cm[a, b] > cm.max() / 2 else "#08306b")
        ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=20, ha="right", fontsize=8)
        ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes, fontsize=8)
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
        ax.set_title(f"{name} (Test Acc = {acc:.2f}%)", fontsize=10, fontweight="bold")
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(title, fontsize=13); fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight"); plt.close(fig)


def run_variant(name, X, y, tr, va, te, classes):
    print("\n" + "=" * 72 + f"\n{name}: {X.shape[1]} features -> {list(X.columns)}\n" + "=" * 72)
    sc = StandardScaler().fit(X.iloc[tr])
    Xtr, Xva, Xte = sc.transform(X.iloc[tr]), sc.transform(X.iloc[va]), sc.transform(X.iloc[te])
    ytr, yva, yte = y[tr], y[va], y[te]
    n_cls = len(classes)
    rows, tune_rows, cms, best_models = [], [], {}, {}

    for mname, (make, grid) in model_grid().items():
        t0 = time.time()
        if mname == "SVM" and len(ytr) > SVM_TUNE_SAMPLE:
            sub, _ = train_test_split(np.arange(len(ytr)), train_size=SVM_TUNE_SAMPLE,
                                      stratify=ytr, random_state=SEED)
        else:
            sub = np.arange(len(ytr))
        best, best_f1 = None, -1
        for params in ParameterGrid(grid):
            m = make(**params)
            if mname == "SVM":
                m.set_params(probability=False)
            m.fit(Xtr[sub], ytr[sub])
            f1v = f1_score(yva, m.predict(Xva), average="macro") * 100
            tune_rows.append({"Variant": name, "Model": mname, "Params": str(params),
                              "Val macro F1": round(f1v, 4)})
            if f1v > best_f1:
                best, best_f1 = params, f1v
        # refit best on full training set, evaluate once on test
        m = make(**best); m.fit(Xtr, ytr)
        tr_m, _ = evaluate(m, Xtr, ytr, n_cls)
        va_m, _ = evaluate(m, Xva, yva, n_cls)
        te_m, pred = evaluate(m, Xte, yte, n_cls)
        cms[mname] = confusion_matrix(yte, pred)
        best_models[mname] = (m, te_m["F1"])
        rows.append({"Variant": name, "Model": mname, "Best params": str(best),
                     "Train Acc": tr_m["acc"], "Val Acc": va_m["acc"], "Val Macro F1": va_m["F1"],
                     "Test Acc": te_m["acc"], "Test Macro P": te_m["P"], "Test Macro R": te_m["R"],
                     "Test Macro F1": te_m["F1"], "Test Weighted F1": te_m["F1w"],
                     "Test ROC-AUC": te_m["AUC"], "Time (s)": time.time() - t0})
        print(f"  {mname:<20s} best {best} | train {tr_m['acc']:6.2f} | val {va_m['acc']:6.2f} | "
              f"TEST acc {te_m['acc']:6.2f}  macro F1 {te_m['F1']:6.2f}  AUC {te_m['AUC']:.4f}")

    plot_cms(cms, list(classes), f"APA-DDoS — {name} — test set", f"{OUT}/{name}_test_confusion.png")

    # permutation importance of the best model (validation set)
    top = max(best_models, key=lambda k: best_models[k][1])
    pi = permutation_importance(best_models[top][0], Xva, yva, n_repeats=5,
                                random_state=SEED, scoring="f1_macro", n_jobs=-1)
    pi_df = pd.DataFrame({"Feature": X.columns, "Importance mean": pi.importances_mean,
                          "Importance sd": pi.importances_std}).sort_values("Importance mean", ascending=False)
    pi_df.insert(0, "Model", top); pi_df.insert(0, "Variant", name)
    print(f"\n  Permutation importance ({top}, validation):")
    print(pi_df.head(8).round(4).to_string(index=False))
    return rows, tune_rows, pi_df


# ---------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------
if __name__ == "__main__":
    df = load()
    le = LabelEncoder(); y = le.fit_transform(df["label"]); classes = le.classes_
    tr, va, te = three_way_split(df)
    print(f"Split: train {len(tr):,} | validation {len(va):,} | test {len(te):,}")

    X, audit, proxy, mi = audit_features(df, tr, va, classes, y)
    audit.to_csv(f"{OUT}/feature_audit.csv", index=False)
    proxy.round(3).to_csv(f"{OUT}/label_proxy_screen.csv", index=False)
    mi.round(4).to_csv(f"{OUT}/mutual_information_ranking.csv", index=False)

    print("FEATURE AUDIT")
    print(audit.to_string(index=False))
    flagged = proxy[proxy["Single-feature proxy"]]
    print("\nSINGLE-FEATURE PROXIES (depth-1 tree >= %.1f%% on validation):" % PROXY_THRESHOLD)
    print(flagged.round(2).to_string(index=False) if len(flagged) else "  none")
    print("\nMUTUAL INFORMATION (train):")
    print(mi.round(4).to_string(index=False))

    subtype_proxies = attack_pair_proxies(proxy, classes)
    variants = {
        "A_LEAKAGE_FREE": X,
        "B_BEHAVIOUR_ONLY": X.drop(columns=subtype_proxies),
    }
    print("\nVariant B removes attack-subtype proxies:", subtype_proxies or "none")

    all_rows, all_tune, all_pi = [], [], []
    for vname, Xv in variants.items():
        r, t, p = run_variant(vname, Xv, y, tr, va, te, classes)
        all_rows += r; all_tune += t; all_pi.append(p)

    res = pd.DataFrame(all_rows).round(4)
    res.to_csv(f"{OUT}/final_results.csv", index=False)
    pd.DataFrame(all_tune).to_csv(f"{OUT}/tuning_on_validation.csv", index=False)
    pd.concat(all_pi).round(4).to_csv(f"{OUT}/permutation_importance.csv", index=False)

    print("\n\nFINAL TEST RESULTS (%)")
    print(res[["Variant", "Model", "Train Acc", "Val Acc", "Test Acc", "Test Macro F1",
               "Test ROC-AUC"]].to_string(index=False))
    print(f"\nAll outputs saved in ./{OUT}/")

Shape: (151200, 23)
Class distribution:
 label
Benign          75600
DDoS-PSH-ACK    37800
DDoS-ACK        37800
Name: count, dtype: int64 

Split: train 90,720 | validation 30,240 | test 30,240
FEATURE AUDIT
          Feature  Status                                              Reason
           ip.src removed           identifier (host/port/time/session field)
           ip.dst removed           identifier (host/port/time/session field)
      tcp.srcport removed           identifier (host/port/time/session field)
      tcp.dstport removed           identifier (host/port/time/session field)
       frame.time removed           identifier (host/port/time/session field)
          tcp.seq removed           identifier (host/port/time/session field)
          tcp.ack removed           identifier (host/port/time/session field)
         ip.proto removed constant/near-constant on train (100.00% one value)
    tcp.flags.syn removed constant/near-constant on train (100.00% one value)
  tcp.flags

In [3]:
# =====================================================================
# APA-DDoS — REVISION EXPERIMENTS v2 (Reviewer comments 1, 2, 3)
#
#   E0   ALL_FEATURES          : same inputs as the submitted paper (Fig. 22)
#   E0b  ALL_FEATURES_TIME_NUM : E0, but frame.time parsed to a number instead
#                                of ordinal-encoded -> shows the E0 tree collapse
#                                is an unseen-category encoding artefact
#   E1   NO_IDENTIFIERS        : IPs, ports, frame.time, tcp.seq, tcp.ack removed
#   E2   NO_ID_NO_PUSH         : E1 + tcp.flags.push removed (label-proxy ablation)
#   E3   BINARY_NO_ID_NO_PUSH  : E2 features, Benign vs DDoS only
#                                (is detection itself trivially separable?)
#
#   Each experiment: "constrained" models (Section 3.2) + "unconstrained" trees.
#
#   Diagnostics:
#     - identifier columns actually found / constant columns dropped
#     - % of TEST rows with a categorical value UNSEEN in training
#     - exact-duplicate rows, and test rows whose feature vector occurs in train
#     - empirical accuracy ceiling from conflicting-label duplicate vectors
#     - single-feature stumps: ACK vs PSH-ACK, and Benign vs DDoS
#     - one-line rule baseline (tcp.flags.push) for the attack subtype
#     - per-class recall for every model
#     - optional repeated seeds (mean +/- SD)
#
#   Outputs -> ./revision_outputs_v2/
# =====================================================================

import os, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder, label_binarize
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, roc_auc_score)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
DATA_PATH = "/kaggle/input/datasets/yashwanthkumbam/apaddos-dataset/APA-DDoS-Dataset/APA-DDoS-Dataset.csv"
OUT = "revision_outputs_v2"
SEEDS = [42]              # e.g. [0, 1, 2, 3, 4] for mean +/- SD (SVM is slow: ~15 min per E2 fit)
RUN_SVM = True
DEDUPLICATE = False       # True = drop exact duplicate rows BEFORE splitting (report both runs)
os.makedirs(OUT, exist_ok=True)

IDENTIFIER_COLS = ["ip.src", "ip.dst", "tcp.srcport", "tcp.dstport",
                   "frame.time", "tcp.seq", "tcp.ack"]
PROXY_COLS = ["tcp.flags.push"]
TIME_COL = "frame.time"
PUSH_COL = "tcp.flags.push"
BENIGN_NAME = "benign"    # matched case-insensitively against label values

EXPERIMENTS = {
    "E0_ALL_FEATURES":           dict(drop=[],                             task="multiclass", time_numeric=False),
    "E0b_ALL_FEATURES_TIME_NUM": dict(drop=[],                             task="multiclass", time_numeric=True),
    "E1_NO_IDENTIFIERS":         dict(drop=IDENTIFIER_COLS,                task="multiclass", time_numeric=False),
    "E2_NO_ID_NO_PUSH":          dict(drop=IDENTIFIER_COLS + PROXY_COLS,   task="multiclass", time_numeric=False),
    "E3_BINARY_NO_ID_NO_PUSH":   dict(drop=IDENTIFIER_COLS + PROXY_COLS,   task="binary",     time_numeric=False),
}


def constrained_models(seed):
    """Hyper-parameters as reported in Section 3.2 of the manuscript.
    NOTE: the paper states solver='liblinear' for LR (one-vs-rest). lbfgs is used
    here because it is the multinomial softmax of Eq. (11). Make the paper and the
    code agree before resubmission."""
    m = {
        "Logistic Regression": LogisticRegression(solver="lbfgs", C=0.03, max_iter=1000,
                                                  random_state=seed),
        "SVM": SVC(kernel="rbf", C=1.0, probability=True, random_state=seed),
        "KNN": KNeighborsClassifier(n_neighbors=5, metric="manhattan", weights="distance",
                                    n_jobs=-1),
        "Decision Tree": DecisionTreeClassifier(criterion="gini", max_depth=8,
                                                max_leaf_nodes=11, random_state=seed),
        "Random Forest": RandomForestClassifier(n_estimators=20, criterion="entropy",
                                                max_depth=10, min_samples_split=100,
                                                min_samples_leaf=20, max_features="sqrt",
                                                random_state=seed, n_jobs=-1),
        "XGBoost": XGBClassifier(eval_metric="mlogloss", random_state=seed, n_jobs=-1),
    }
    if not RUN_SVM:
        m.pop("SVM")
    return m


def unconstrained_trees(seed):
    return {
        "Decision Tree (unconstrained)": DecisionTreeClassifier(random_state=seed),
        "Random Forest (unconstrained)": RandomForestClassifier(n_estimators=200,
                                                                random_state=seed, n_jobs=-1),
        "XGBoost (500 trees, depth 10)": XGBClassifier(n_estimators=500, max_depth=10,
                                                       eval_metric="mlogloss",
                                                       random_state=seed, n_jobs=-1),
    }


# ---------------------------------------------------------------------
# DATA
# ---------------------------------------------------------------------
def load():
    df = pd.read_csv(DATA_PATH, low_memory=False)
    df.columns = df.columns.str.strip().str.lower()
    print("Columns in file:", list(df.columns))
    print("Shape:", df.shape)
    n_dup = int(df.duplicated().sum())
    n_dup_x = int(df.drop(columns=["label"]).duplicated().sum())
    print(f"Exact duplicate rows (incl. label): {n_dup:,}  |  duplicate feature vectors: {n_dup_x:,}")
    if DEDUPLICATE:
        df = df.drop_duplicates().reset_index(drop=True)
        print("DEDUPLICATE=True -> shape after dropping duplicates:", df.shape)
    return df


def make_labels(y_raw, task):
    if task == "binary":
        y_raw = np.where(y_raw.astype(str).str.lower() == BENIGN_NAME, "Benign", "DDoS")
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    return y, le.classes_


def split(df, drop_cols, task, time_numeric, seed):
    X = df.drop(columns=["label"])
    found = [c for c in drop_cols if c in X.columns]
    missing = [c for c in drop_cols if c not in X.columns]
    X = X.drop(columns=found)

    if time_numeric and TIME_COL in X.columns:
        t = pd.to_datetime(X[TIME_COL], errors="coerce")
        print(f"  {TIME_COL} parsed to numeric: {t.isna().mean()*100:.2f}% unparseable")
        X[TIME_COL] = (t - t.min()).dt.total_seconds()

    y, classes = make_labels(df["label"], task)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.30,
                                              random_state=seed, stratify=y)
    X_tr, X_te = X_tr.copy(), X_te.copy()

    cat_cols = list(X_tr.select_dtypes(include=["object", "category"]).columns)
    unseen = {}
    if cat_cols:
        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        X_tr[cat_cols] = enc.fit_transform(X_tr[cat_cols].astype(str))
        X_te[cat_cols] = enc.transform(X_te[cat_cols].astype(str))
        for c in cat_cols:
            unseen[c] = float((X_te[c] == -1).mean() * 100)

    X_tr = X_tr.apply(pd.to_numeric, errors="coerce")
    X_te = X_te.apply(pd.to_numeric, errors="coerce")
    med = X_tr.median()
    X_tr, X_te = X_tr.fillna(med), X_te.fillna(med)

    const = [c for c in X_tr.columns if X_tr[c].nunique() <= 1]
    X_tr, X_te = X_tr.drop(columns=const), X_te.drop(columns=const)

    feats = list(X_tr.columns)
    sc = StandardScaler()
    X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)
    info = {"features": feats, "categorical": cat_cols, "constant_dropped": const,
            "unseen_pct": unseen, "drop_found": found, "drop_missing": missing,
            "raw_train": X_tr, "raw_test": X_te}
    return X_tr_s, X_te_s, y_tr, y_te, classes, info


# ---------------------------------------------------------------------
# DIAGNOSTICS
# ---------------------------------------------------------------------
def duplicate_audit(X_tr, X_te, y_te):
    """Test rows whose exact feature vector appears in train, and the empirical
    accuracy ceiling implied by identical vectors carrying different labels."""
    key = lambda D: pd.util.hash_pandas_object(D.round(9), index=False)
    h_tr, h_te = set(key(X_tr)), key(X_te)
    overlap = float(h_te.isin(h_tr).mean() * 100)
    d = pd.DataFrame({"h": h_te.values, "y": y_te})
    ceiling = d.groupby("h")["y"].agg(lambda s: s.value_counts().iloc[0]).sum() / len(d) * 100
    conflicting = int((d.groupby("h")["y"].nunique() > 1).sum())
    return {"test_rows_seen_in_train_%": overlap,
            "empirical_ceiling_acc_%": float(ceiling),
            "conflicting_vectors": conflicting}


def class_index(classes, pred):
    return [i for i, c in enumerate(classes) if pred(str(c).lower())]


def stump_audit(X_tr, X_te, y_tr, y_te, classes, feats, pair_name, pos_idx, neg_idx):
    """Depth-1 tree on ONE feature for a two-group problem.
    ~100% from a single feature = that feature alone determines the grouping."""
    keep = pos_idx + neg_idx
    mtr, mte = np.isin(y_tr, keep), np.isin(y_te, keep)
    ytr_b, yte_b = np.isin(y_tr[mtr], pos_idx), np.isin(y_te[mte], pos_idx)
    rows = []
    for j, f in enumerate(feats):
        s = DecisionTreeClassifier(max_depth=1, random_state=0)
        s.fit(X_tr[mtr][:, [j]], ytr_b)
        rows.append({"Task": pair_name, "Feature": f,
                     "Stump Train Acc": s.score(X_tr[mtr][:, [j]], ytr_b) * 100,
                     "Stump Test Acc": s.score(X_te[mte][:, [j]], yte_b) * 100})
    return pd.DataFrame(rows).sort_values("Stump Test Acc", ascending=False)


def push_rule_baseline(raw_te, y_te, classes):
    """if tcp.flags.push set -> PSH-ACK else ACK, scored on attack rows only."""
    if PUSH_COL not in raw_te.columns:
        return None
    ack = class_index(classes, lambda c: "ack" in c and "psh" not in c)
    psh = class_index(classes, lambda c: "psh" in c)
    if not ack or not psh:
        return None
    m = np.isin(y_te, [ack[0], psh[0]])
    push = raw_te[PUSH_COL].values[m]
    thr = np.median(np.unique(push)) if len(np.unique(push)) > 1 else 0
    pred = np.where(push > thr, psh[0], ack[0])
    return float((pred == y_te[m]).mean() * 100)


def importances(model, feats, k=8):
    if hasattr(model, "feature_importances_"):
        return pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False).head(k)
    return None


# ---------------------------------------------------------------------
# METRICS / PLOTS
# ---------------------------------------------------------------------
def evaluate(name, model, X_tr, X_te, y_tr, y_te, classes):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    t_train = time.time() - t0
    y_pred = model.predict(X_te)
    cm = confusion_matrix(y_te, y_pred, labels=np.arange(len(classes)))
    row = {"Model": name,
           "Train Acc": accuracy_score(y_tr, model.predict(X_tr)) * 100,
           "Test Acc": accuracy_score(y_te, y_pred) * 100}
    for avg in ["macro", "weighted"]:
        p, r, f, _ = precision_recall_fscore_support(y_te, y_pred, average=avg, zero_division=0)
        row[f"P {avg}"], row[f"R {avg}"], row[f"F1 {avg}"] = p * 100, r * 100, f * 100
    for i, c in enumerate(classes):
        row[f"Recall {c}"] = cm[i, i] / max(cm[i].sum(), 1) * 100
    try:
        prob = model.predict_proba(X_te)
        if len(classes) == 2:
            row["ROC-AUC"] = roc_auc_score(y_te, prob[:, 1])
        else:
            row["ROC-AUC"] = roc_auc_score(label_binarize(y_te, classes=np.arange(len(classes))),
                                           prob, multi_class="ovr", average="macro")
    except Exception as e:
        print("  AUC failed:", e)
        row["ROC-AUC"] = np.nan
    row["Train Time (s)"] = t_train
    return row, cm


def plot_cms(cms, classes, title, path):
    n, cols = len(cms), 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.3 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (name, (cm, acc)) in zip(axes, cms.items()):
        ax.imshow(cm, cmap="Blues")
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black",
                        fontsize=10, fontweight="bold")
        ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=20, ha="right", fontsize=8)
        ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes, fontsize=8)
        ax.set_title(f"{name} (Acc = {acc:.2f}%)", fontsize=10, fontweight="bold")
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)


# ---------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------
if __name__ == "__main__":
    df = load()
    print("\nClass distribution:\n", df["label"].value_counts())

    all_rows, feature_tables, diag_rows, stump_tables = [], [], [], []

    for seed in SEEDS:
        for exp, cfg in EXPERIMENTS.items():
            print("\n" + "=" * 70 + f"\n{exp}  seed={seed}  (drop: {cfg['drop'] or 'nothing'})\n" + "=" * 70)
            X_tr, X_te, y_tr, y_te, classes, info = split(df, cfg["drop"], cfg["task"],
                                                          cfg["time_numeric"], seed)
            feats = info["features"]
            print(f"  {len(feats)} input features used:", feats)
            print("  Drop-list columns FOUND and removed:", info["drop_found"])
            if info["drop_missing"]:
                print("  Drop-list columns NOT in file:", info["drop_missing"])
            print("  Constant columns dropped:", info["constant_dropped"])
            for c, v in info["unseen_pct"].items():
                print(f"  UNSEEN in train -> {c:<20s} {v:6.2f}% of test rows")

            dup = duplicate_audit(info["raw_train"], info["raw_test"], y_te)
            rule = push_rule_baseline(info["raw_test"], y_te, classes)
            print(f"  Test rows whose feature vector is in train: {dup['test_rows_seen_in_train_%']:.2f}%")
            print(f"  Empirical accuracy ceiling (conflicting duplicates): {dup['empirical_ceiling_acc_%']:.2f}%"
                  f"  [{dup['conflicting_vectors']:,} conflicting vectors]")
            if rule is not None:
                print(f"  Rule baseline 'push flag -> PSH-ACK' (attack rows): {rule:.2f}%")
            diag_rows.append({"Experiment": exp, "Seed": seed, "N features": len(feats),
                              **dup, "push_rule_subtype_acc_%": rule,
                              **{f"unseen_{c}_%": v for c, v in info["unseen_pct"].items()}})
            if seed == SEEDS[0]:
                feature_tables.append(pd.DataFrame({"Experiment": exp, "Feature": feats}))

                ben = class_index(classes, lambda c: c == BENIGN_NAME)
                att = [i for i in range(len(classes)) if i not in ben]
                if ben and att:
                    stump_tables.append(stump_audit(X_tr, X_te, y_tr, y_te, classes, feats,
                                                    "Benign vs DDoS", ben, att).assign(Experiment=exp))
                ack = class_index(classes, lambda c: "ack" in c and "psh" not in c)
                psh = class_index(classes, lambda c: "psh" in c)
                if ack and psh:
                    stump_tables.append(stump_audit(X_tr, X_te, y_tr, y_te, classes, feats,
                                                    "ACK vs PSH-ACK", psh[:1], ack[:1]).assign(Experiment=exp))
                for st in stump_tables[-2:]:
                    if st["Experiment"].iloc[0] == exp:
                        print(f"\n  Top single-feature stumps ({st['Task'].iloc[0]}):")
                        print(st.head(4)[["Feature", "Stump Train Acc", "Stump Test Acc"]].to_string(index=False))

            for setting, models in {"constrained": constrained_models(seed),
                                    "unconstrained": unconstrained_trees(seed)}.items():
                cms = {}
                for name, model in models.items():
                    row, cm = evaluate(name, model, X_tr, X_te, y_tr, y_te, classes)
                    row.update({"Experiment": exp, "Setting": setting, "Seed": seed,
                                "N features": len(feats)})
                    all_rows.append(row)
                    cms[name] = (cm, row["Test Acc"])
                    rec = " | ".join(f"{c} {row[f'Recall {c}']:.1f}" for c in classes)
                    print(f"  [{setting}] {name:<32s} train {row['Train Acc']:.2f} | test {row['Test Acc']:.2f}"
                          f" | F1-macro {row['F1 macro']:.2f} | recall: {rec}")
                    imp = importances(model, feats)
                    if imp is not None:
                        print("      top importances:", ", ".join(f"{k}={v:.3f}" for k, v in imp.items()))
                if seed == SEEDS[0]:
                    plot_cms(cms, list(classes), f"APA-DDoS — {exp} — {setting}",
                             f"{OUT}/{exp}_{setting}_confusion.png")

    res = pd.DataFrame(all_rows)
    lead = ["Experiment", "Setting", "Seed", "N features", "Model", "Train Acc", "Test Acc",
            "P macro", "R macro", "F1 macro", "P weighted", "R weighted", "F1 weighted"]
    recall_cols = sorted(c for c in res.columns if c.startswith("Recall "))
    res = res[lead + recall_cols + ["ROC-AUC", "Train Time (s)"]].round(4)
    res.to_csv(f"{OUT}/all_results.csv", index=False)
    pd.DataFrame(diag_rows).round(4).to_csv(f"{OUT}/diagnostics.csv", index=False)
    pd.concat(feature_tables).to_csv(f"{OUT}/features_used_per_experiment.csv", index=False)
    if stump_tables:
        pd.concat(stump_tables).round(4).to_csv(f"{OUT}/single_feature_stumps.csv", index=False)

    if len(SEEDS) > 1:
        agg = (res.groupby(["Experiment", "Setting", "Model"])[["Test Acc", "F1 macro", "ROC-AUC"]]
                  .agg(["mean", "std"]).round(2))
        agg.to_csv(f"{OUT}/results_mean_sd.csv")
        print("\nMEAN +/- SD over seeds\n", agg)

    print("\n\nSUMMARY (test accuracy %, first seed)")
    print(res[res["Seed"] == SEEDS[0]].pivot_table(index="Model", columns="Experiment",
                                                   values="Test Acc").round(2))
    print(f"\nAll outputs saved in ./{OUT}/")

Columns in file: ['ip.src', 'ip.dst', 'tcp.srcport', 'tcp.dstport', 'ip.proto', 'frame.len', 'tcp.flags.syn', 'tcp.flags.reset', 'tcp.flags.push', 'tcp.flags.ack', 'ip.flags.mf', 'ip.flags.df', 'ip.flags.rb', 'tcp.seq', 'tcp.ack', 'frame.time', 'packets', 'bytes', 'tx packets', 'tx bytes', 'rx packets', 'rx bytes', 'label']
Shape: (151200, 23)
Exact duplicate rows (incl. label): 0  |  duplicate feature vectors: 0

Class distribution:
 label
Benign          75600
DDoS-PSH-ACK    37800
DDoS-ACK        37800
Name: count, dtype: int64

E0_ALL_FEATURES  seed=42  (drop: nothing)
  12 input features used: ['ip.src', 'tcp.srcport', 'frame.len', 'tcp.flags.push', 'ip.flags.df', 'frame.time', 'packets', 'bytes', 'tx packets', 'tx bytes', 'rx packets', 'rx bytes']
  Drop-list columns FOUND and removed: []
  Constant columns dropped: ['ip.dst', 'tcp.dstport', 'ip.proto', 'tcp.flags.syn', 'tcp.flags.reset', 'tcp.flags.ack', 'ip.flags.mf', 'ip.flags.rb', 'tcp.seq', 'tcp.ack']
  UNSEEN in train -> ip

In [4]:
# =====================================================================
# APA-DDoS — REVISION EXPERIMENTS v2 (Reviewer comments 1, 2, 3)
#
#   E0   ALL_FEATURES          : same inputs as the submitted paper (Fig. 22)
#   E0b  ALL_FEATURES_TIME_NUM : E0, but frame.time parsed to a number instead
#                                of ordinal-encoded -> shows the E0 tree collapse
#                                is an unseen-category encoding artefact
#   E1   NO_IDENTIFIERS        : IPs, ports, frame.time, tcp.seq, tcp.ack removed
#   E2   NO_ID_NO_PUSH         : E1 + tcp.flags.push removed (label-proxy ablation)
#   E3   BINARY_NO_ID_NO_PUSH  : E2 features, Benign vs DDoS only
#                                (is detection itself trivially separable?)
#
#   Each experiment: "constrained" models (Section 3.2) + "unconstrained" trees.
#
#   Diagnostics:
#     - identifier columns actually found / constant columns dropped
#     - % of TEST rows with a categorical value UNSEEN in training
#     - exact-duplicate rows, and test rows whose feature vector occurs in train
#     - empirical accuracy ceiling from conflicting-label duplicate vectors
#     - single-feature stumps: ACK vs PSH-ACK, and Benign vs DDoS
#     - one-line rule baseline (tcp.flags.push) for the attack subtype
#     - per-class recall for every model
#     - optional repeated seeds (mean +/- SD)
#
#   Outputs -> ./revision_outputs_v2/
# =====================================================================

import os, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder, label_binarize
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, roc_auc_score)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
DATA_PATH = "/kaggle/input/datasets/yashwanthkumbam/apaddos-dataset/APA-DDoS-Dataset/APA-DDoS-Dataset.csv"
OUT = "revision_outputs_v2"
SEEDS = [42]              # e.g. [0, 1, 2, 3, 4] for mean +/- SD (SVM is slow: ~15 min per E2 fit)
RUN_SVM = True
DEDUPLICATE = False       # True = drop exact duplicate rows BEFORE splitting (report both runs)
os.makedirs(OUT, exist_ok=True)

IDENTIFIER_COLS = ["ip.src", "ip.dst", "tcp.srcport", "tcp.dstport",
                   "frame.time", "tcp.seq", "tcp.ack"]
PROXY_COLS = ["tcp.flags.push"]
TIME_COL = "frame.time"
PUSH_COL = "tcp.flags.push"
BENIGN_NAME = "benign"    # matched case-insensitively against label values

EXPERIMENTS = {
    "E0_ALL_FEATURES":           dict(drop=[],                             task="multiclass", time_numeric=False),
    "E0b_ALL_FEATURES_TIME_NUM": dict(drop=[],                             task="multiclass", time_numeric=True),
    "E1_NO_IDENTIFIERS":         dict(drop=IDENTIFIER_COLS,                task="multiclass", time_numeric=False),
    "E2_NO_ID_NO_PUSH":          dict(drop=IDENTIFIER_COLS + PROXY_COLS,   task="multiclass", time_numeric=False),
    "E3_BINARY_NO_ID_NO_PUSH":   dict(drop=IDENTIFIER_COLS + PROXY_COLS,   task="binary",     time_numeric=False),
}


def constrained_models(seed):
    """Hyper-parameters as reported in Section 3.2 of the manuscript.
    NOTE: the paper states solver='liblinear' for LR (one-vs-rest). lbfgs is used
    here because it is the multinomial softmax of Eq. (11). Make the paper and the
    code agree before resubmission."""
    m = {
        "Logistic Regression": LogisticRegression(solver="lbfgs", C=0.03, max_iter=1000,
                                                  random_state=seed),
        "SVM": SVC(kernel="rbf", C=1.0, probability=True, random_state=seed),
        "KNN": KNeighborsClassifier(n_neighbors=5, metric="manhattan", weights="distance",
                                    n_jobs=-1),
        "Decision Tree": DecisionTreeClassifier(criterion="gini", max_depth=8,
                                                max_leaf_nodes=11, random_state=seed),
        "Random Forest": RandomForestClassifier(n_estimators=20, criterion="entropy",
                                                max_depth=10, min_samples_split=100,
                                                min_samples_leaf=20, max_features="sqrt",
                                                random_state=seed, n_jobs=-1),
        "XGBoost": XGBClassifier(eval_metric="mlogloss", random_state=seed, n_jobs=-1),
    }
    if not RUN_SVM:
        m.pop("SVM")
    return m


def unconstrained_trees(seed):
    return {
        "Decision Tree (unconstrained)": DecisionTreeClassifier(random_state=seed),
        "Random Forest (unconstrained)": RandomForestClassifier(n_estimators=200,
                                                                random_state=seed, n_jobs=-1),
        "XGBoost (500 trees, depth 10)": XGBClassifier(n_estimators=500, max_depth=10,
                                                       eval_metric="mlogloss",
                                                       random_state=seed, n_jobs=-1),
    }


# ---------------------------------------------------------------------
# DATA
# ---------------------------------------------------------------------
def load():
    df = pd.read_csv(DATA_PATH, low_memory=False)
    df.columns = df.columns.str.strip().str.lower()
    print("Columns in file:", list(df.columns))
    print("Shape:", df.shape)
    n_dup = int(df.duplicated().sum())
    n_dup_x = int(df.drop(columns=["label"]).duplicated().sum())
    print(f"Exact duplicate rows (incl. label): {n_dup:,}  |  duplicate feature vectors: {n_dup_x:,}")
    if DEDUPLICATE:
        df = df.drop_duplicates().reset_index(drop=True)
        print("DEDUPLICATE=True -> shape after dropping duplicates:", df.shape)
    return df


def make_labels(y_raw, task):
    if task == "binary":
        y_raw = np.where(y_raw.astype(str).str.lower() == BENIGN_NAME, "Benign", "DDoS")
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    return y, le.classes_


def split(df, drop_cols, task, time_numeric, seed):
    X = df.drop(columns=["label"])
    found = [c for c in drop_cols if c in X.columns]
    missing = [c for c in drop_cols if c not in X.columns]
    X = X.drop(columns=found)

    if time_numeric and TIME_COL in X.columns:
        s = X[TIME_COL].astype(str).str.strip()
        print(f"  {TIME_COL} raw examples: {s.iloc[:2].tolist()}")
        # tshark writes e.g. "Jun 16, 2020 20:18:15.071112000 India Standard Time"
        # or "16-Jun 2020 20:18:15.071112000": keep date+time, trim ns -> us, drop zone text
        core = s.str.extract(
            r"([A-Za-z]{3}\s+\d{1,2},\s+\d{4}\s+\d{1,2}:\d{2}:\d{2}(?:\.\d+)?"
            r"|\d{1,2}-[A-Za-z]{3}[\s-]\d{4}\s+\d{1,2}:\d{2}:\d{2}(?:\.\d+)?"
            r"|\d{4}-\d{2}-\d{2}[ T]\d{1,2}:\d{2}:\d{2}(?:\.\d+)?)")[0]
        core = core.str.replace(r"(\.\d{6})\d+", r"\1", regex=True)
        t = pd.to_datetime(core, errors="coerce", format="mixed")
        print(f"  {TIME_COL} parsed to numeric: {t.isna().mean()*100:.2f}% unparseable")
        X[TIME_COL] = (t - t.min()).dt.total_seconds()

    y, classes = make_labels(df["label"], task)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.30,
                                              random_state=seed, stratify=y)
    X_tr, X_te = X_tr.copy(), X_te.copy()

    cat_cols = list(X_tr.select_dtypes(include=["object", "category"]).columns)
    unseen = {}
    if cat_cols:
        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        X_tr[cat_cols] = enc.fit_transform(X_tr[cat_cols].astype(str))
        X_te[cat_cols] = enc.transform(X_te[cat_cols].astype(str))
        for c in cat_cols:
            unseen[c] = float((X_te[c] == -1).mean() * 100)

    X_tr = X_tr.apply(pd.to_numeric, errors="coerce")
    X_te = X_te.apply(pd.to_numeric, errors="coerce")
    med = X_tr.median()
    X_tr, X_te = X_tr.fillna(med), X_te.fillna(med)

    const = [c for c in X_tr.columns if X_tr[c].nunique() <= 1]
    X_tr, X_te = X_tr.drop(columns=const), X_te.drop(columns=const)

    feats = list(X_tr.columns)
    sc = StandardScaler()
    X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)
    info = {"features": feats, "categorical": cat_cols, "constant_dropped": const,
            "unseen_pct": unseen, "drop_found": found, "drop_missing": missing,
            "raw_train": X_tr, "raw_test": X_te}
    return X_tr_s, X_te_s, y_tr, y_te, classes, info


# ---------------------------------------------------------------------
# DIAGNOSTICS
# ---------------------------------------------------------------------
def duplicate_audit(X_tr, X_te, y_te):
    """Test rows whose exact feature vector appears in train, and the empirical
    accuracy ceiling implied by identical vectors carrying different labels."""
    key = lambda D: pd.util.hash_pandas_object(D.round(9), index=False)
    h_tr, h_te = set(key(X_tr)), key(X_te)
    overlap = float(h_te.isin(h_tr).mean() * 100)
    d = pd.DataFrame({"h": h_te.values, "y": y_te})
    ceiling = d.groupby("h")["y"].agg(lambda s: s.value_counts().iloc[0]).sum() / len(d) * 100
    conflicting = int((d.groupby("h")["y"].nunique() > 1).sum())
    h_all = pd.concat([key(X_tr), h_te], ignore_index=True)
    return {"test_rows_seen_in_train_%": overlap,
            "empirical_ceiling_acc_%": float(ceiling),
            "conflicting_vectors": conflicting,
            "unique_feature_vectors": int(h_all.nunique()),
            "total_rows": int(len(h_all))}


def class_index(classes, pred):
    return [i for i, c in enumerate(classes) if pred(str(c).lower())]


def stump_audit(X_tr, X_te, y_tr, y_te, classes, feats, pair_name, pos_idx, neg_idx):
    """Depth-1 tree on ONE feature for a two-group problem.
    ~100% from a single feature = that feature alone determines the grouping."""
    keep = pos_idx + neg_idx
    mtr, mte = np.isin(y_tr, keep), np.isin(y_te, keep)
    ytr_b, yte_b = np.isin(y_tr[mtr], pos_idx), np.isin(y_te[mte], pos_idx)
    rows = []
    for j, f in enumerate(feats):
        s = DecisionTreeClassifier(max_depth=1, random_state=0)
        s.fit(X_tr[mtr][:, [j]], ytr_b)
        rows.append({"Task": pair_name, "Feature": f,
                     "Stump Train Acc": s.score(X_tr[mtr][:, [j]], ytr_b) * 100,
                     "Stump Test Acc": s.score(X_te[mte][:, [j]], yte_b) * 100})
    return pd.DataFrame(rows).sort_values("Stump Test Acc", ascending=False)


def push_rule_baseline(raw_te, y_te, classes):
    """if tcp.flags.push set -> PSH-ACK else ACK, scored on attack rows only."""
    if PUSH_COL not in raw_te.columns:
        return None
    ack = class_index(classes, lambda c: "ack" in c and "psh" not in c)
    psh = class_index(classes, lambda c: "psh" in c)
    if not ack or not psh:
        return None
    m = np.isin(y_te, [ack[0], psh[0]])
    push = raw_te[PUSH_COL].values[m]
    thr = np.median(np.unique(push)) if len(np.unique(push)) > 1 else 0
    pred = np.where(push > thr, psh[0], ack[0])
    return float((pred == y_te[m]).mean() * 100)


def importances(model, feats, k=8):
    if hasattr(model, "feature_importances_"):
        return pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False).head(k)
    return None


# ---------------------------------------------------------------------
# METRICS / PLOTS
# ---------------------------------------------------------------------
def evaluate(name, model, X_tr, X_te, y_tr, y_te, classes):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    t_train = time.time() - t0
    y_pred = model.predict(X_te)
    cm = confusion_matrix(y_te, y_pred, labels=np.arange(len(classes)))
    row = {"Model": name,
           "Train Acc": accuracy_score(y_tr, model.predict(X_tr)) * 100,
           "Test Acc": accuracy_score(y_te, y_pred) * 100}
    for avg in ["macro", "weighted"]:
        p, r, f, _ = precision_recall_fscore_support(y_te, y_pred, average=avg, zero_division=0)
        row[f"P {avg}"], row[f"R {avg}"], row[f"F1 {avg}"] = p * 100, r * 100, f * 100
    for i, c in enumerate(classes):
        row[f"Recall {c}"] = cm[i, i] / max(cm[i].sum(), 1) * 100
    try:
        prob = model.predict_proba(X_te)
        if len(classes) == 2:
            row["ROC-AUC"] = roc_auc_score(y_te, prob[:, 1])
        else:
            row["ROC-AUC"] = roc_auc_score(label_binarize(y_te, classes=np.arange(len(classes))),
                                           prob, multi_class="ovr", average="macro")
    except Exception as e:
        print("  AUC failed:", e)
        row["ROC-AUC"] = np.nan
    row["Train Time (s)"] = t_train
    return row, cm


def plot_cms(cms, classes, title, path):
    n, cols = len(cms), 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.3 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (name, (cm, acc)) in zip(axes, cms.items()):
        ax.imshow(cm, cmap="Blues")
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black",
                        fontsize=10, fontweight="bold")
        ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=20, ha="right", fontsize=8)
        ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes, fontsize=8)
        ax.set_title(f"{name} (Acc = {acc:.2f}%)", fontsize=10, fontweight="bold")
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)


# ---------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------
if __name__ == "__main__":
    df = load()
    print("\nClass distribution:\n", df["label"].value_counts())

    all_rows, feature_tables, diag_rows, stump_tables = [], [], [], []

    for seed in SEEDS:
        for exp, cfg in EXPERIMENTS.items():
            print("\n" + "=" * 70 + f"\n{exp}  seed={seed}  (drop: {cfg['drop'] or 'nothing'})\n" + "=" * 70)
            X_tr, X_te, y_tr, y_te, classes, info = split(df, cfg["drop"], cfg["task"],
                                                          cfg["time_numeric"], seed)
            feats = info["features"]
            print(f"  {len(feats)} input features used:", feats)
            print("  Drop-list columns FOUND and removed:", info["drop_found"])
            if info["drop_missing"]:
                print("  Drop-list columns NOT in file:", info["drop_missing"])
            print("  Constant columns dropped:", info["constant_dropped"])
            for c, v in info["unseen_pct"].items():
                print(f"  UNSEEN in train -> {c:<20s} {v:6.2f}% of test rows")

            dup = duplicate_audit(info["raw_train"], info["raw_test"], y_te)
            rule = push_rule_baseline(info["raw_test"], y_te, classes)
            print(f"  Test rows whose feature vector is in train: {dup['test_rows_seen_in_train_%']:.2f}%")
            print(f"  Distinct feature vectors: {dup['unique_feature_vectors']:,} of {dup['total_rows']:,} rows")
            print(f"  Empirical accuracy ceiling (conflicting duplicates): {dup['empirical_ceiling_acc_%']:.2f}%"
                  f"  [{dup['conflicting_vectors']:,} conflicting vectors]")
            if rule is not None:
                print(f"  Rule baseline 'push flag -> PSH-ACK' (attack rows): {rule:.2f}%")
            diag_rows.append({"Experiment": exp, "Seed": seed, "N features": len(feats),
                              **dup, "push_rule_subtype_acc_%": rule,
                              **{f"unseen_{c}_%": v for c, v in info["unseen_pct"].items()}})
            if seed == SEEDS[0]:
                feature_tables.append(pd.DataFrame({"Experiment": exp, "Feature": feats}))

                ben = class_index(classes, lambda c: c == BENIGN_NAME)
                att = [i for i in range(len(classes)) if i not in ben]
                if ben and att:
                    stump_tables.append(stump_audit(X_tr, X_te, y_tr, y_te, classes, feats,
                                                    "Benign vs DDoS", ben, att).assign(Experiment=exp))
                ack = class_index(classes, lambda c: "ack" in c and "psh" not in c)
                psh = class_index(classes, lambda c: "psh" in c)
                if ack and psh:
                    stump_tables.append(stump_audit(X_tr, X_te, y_tr, y_te, classes, feats,
                                                    "ACK vs PSH-ACK", psh[:1], ack[:1]).assign(Experiment=exp))
                for st in stump_tables[-2:]:
                    if st["Experiment"].iloc[0] == exp:
                        print(f"\n  Top single-feature stumps ({st['Task'].iloc[0]}):")
                        print(st.head(4)[["Feature", "Stump Train Acc", "Stump Test Acc"]].to_string(index=False))

            for setting, models in {"constrained": constrained_models(seed),
                                    "unconstrained": unconstrained_trees(seed)}.items():
                cms = {}
                for name, model in models.items():
                    row, cm = evaluate(name, model, X_tr, X_te, y_tr, y_te, classes)
                    row.update({"Experiment": exp, "Setting": setting, "Seed": seed,
                                "N features": len(feats)})
                    all_rows.append(row)
                    cms[name] = (cm, row["Test Acc"])
                    rec = " | ".join(f"{c} {row[f'Recall {c}']:.1f}" for c in classes)
                    print(f"  [{setting}] {name:<32s} train {row['Train Acc']:.2f} | test {row['Test Acc']:.2f}"
                          f" | F1-macro {row['F1 macro']:.2f} | recall: {rec}")
                    imp = importances(model, feats)
                    if imp is not None:
                        print("      top importances:", ", ".join(f"{k}={v:.3f}" for k, v in imp.items()))
                if seed == SEEDS[0]:
                    plot_cms(cms, list(classes), f"APA-DDoS — {exp} — {setting}",
                             f"{OUT}/{exp}_{setting}_confusion.png")

    res = pd.DataFrame(all_rows)
    lead = ["Experiment", "Setting", "Seed", "N features", "Model", "Train Acc", "Test Acc",
            "P macro", "R macro", "F1 macro", "P weighted", "R weighted", "F1 weighted"]
    recall_cols = sorted(c for c in res.columns if c.startswith("Recall "))
    res = res[lead + recall_cols + ["ROC-AUC", "Train Time (s)"]].round(4)
    res.to_csv(f"{OUT}/all_results.csv", index=False)
    pd.DataFrame(diag_rows).round(4).to_csv(f"{OUT}/diagnostics.csv", index=False)
    pd.concat(feature_tables).to_csv(f"{OUT}/features_used_per_experiment.csv", index=False)
    if stump_tables:
        pd.concat(stump_tables).round(4).to_csv(f"{OUT}/single_feature_stumps.csv", index=False)

    if len(SEEDS) > 1:
        agg = (res.groupby(["Experiment", "Setting", "Model"])[["Test Acc", "F1 macro", "ROC-AUC"]]
                  .agg(["mean", "std"]).round(2))
        agg.to_csv(f"{OUT}/results_mean_sd.csv")
        print("\nMEAN +/- SD over seeds\n", agg)

    print("\n\nSUMMARY (test accuracy %, first seed)")
    print(res[res["Seed"] == SEEDS[0]].pivot_table(index="Model", columns="Experiment",
                                                   values="Test Acc").round(2))
    print(f"\nAll outputs saved in ./{OUT}/")

Columns in file: ['ip.src', 'ip.dst', 'tcp.srcport', 'tcp.dstport', 'ip.proto', 'frame.len', 'tcp.flags.syn', 'tcp.flags.reset', 'tcp.flags.push', 'tcp.flags.ack', 'ip.flags.mf', 'ip.flags.df', 'ip.flags.rb', 'tcp.seq', 'tcp.ack', 'frame.time', 'packets', 'bytes', 'tx packets', 'tx bytes', 'rx packets', 'rx bytes', 'label']
Shape: (151200, 23)
Exact duplicate rows (incl. label): 0  |  duplicate feature vectors: 0

Class distribution:
 label
Benign          75600
DDoS-PSH-ACK    37800
DDoS-ACK        37800
Name: count, dtype: int64

E0_ALL_FEATURES  seed=42  (drop: nothing)
  12 input features used: ['ip.src', 'tcp.srcport', 'frame.len', 'tcp.flags.push', 'ip.flags.df', 'frame.time', 'packets', 'bytes', 'tx packets', 'tx bytes', 'rx packets', 'rx bytes']
  Drop-list columns FOUND and removed: []
  Constant columns dropped: ['ip.dst', 'tcp.dstport', 'ip.proto', 'tcp.flags.syn', 'tcp.flags.reset', 'tcp.flags.ack', 'ip.flags.mf', 'ip.flags.rb', 'tcp.seq', 'tcp.ack']
  UNSEEN in train -> ip